#  Deduplication & Fuzzy Matching - Merging Customer Records

## The Problem Statement

### What the Client Says:
"I have two customer lists from different systems. I need to merge them, but there are duplicates and the names don't match exactly."

### The Deeper Business Pain:
- **Lost Revenue:** Sending marketing emails to the same customer twice wastes budget
- **Bad Analytics:** Customer counts are wrong - you think you have 200 customers but really have 150
- **Embarrassing Mistakes:** Customer service can't see full history because records are split across systems
- **Wasted Time:** Staff manually checking "Is this the same customer?" every day

### The Data Generating Process (DGP):
*Why are there duplicates?*
- **System Migration:** Legacy system had different format (Last, First vs First Last)
- **Manual Entry:** Typos happen when humans type names
- **No Unique ID:** Different systems generated different customer IDs
- **Mergers:** Company acquired another business, combining databases

In [2]:
!pip install thefuzz

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 699.0 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.5 MB 745.8 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.5 MB 786.4 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 786.4 kB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 838.9 kB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 834.0 kB/s eta 0:00:00


In [3]:
# Import our tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from thefuzz import fuzz, process
import re

# Set options for better display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

print("✅ Libraries imported!")

✅ Libraries imported!


In [5]:
# Load both CRM datasets
df_main = pd.read_csv(r"C:\Users\Peter_Chiputah\Desktop\data-cleaning-skills-matrix\data\raw\02_crm_legacy_system.csv")
df_legacy = pd.read_csv(r"C:\Users\Peter_Chiputah\Desktop\data-cleaning-skills-matrix\data\raw\02_crm_main.csv")

print("📊 MAIN CRM SYSTEM:")
print(f"   Records: {len(df_main)}")
print(f"   Columns: {df_main.columns.tolist()}")
print("\n📊 LEGACY SYSTEM:")
print(f"   Records: {len(df_legacy)}")
print(f"   Columns: {df_legacy.columns.tolist()}")

📊 MAIN CRM SYSTEM:
   Records: 140
   Columns: ['customer_id', 'full_name', 'email', 'phone', 'city', 'signup_date']

📊 LEGACY SYSTEM:
   Records: 100
   Columns: ['customer_id', 'full_name', 'email', 'phone', 'city', 'signup_date']


In [6]:
# Look at the data
print("🔍 MAIN CRM - First 5 rows:")
df_main.head()

🔍 MAIN CRM - First 5 rows:


,customer_id,full_name,email,phone,city,signup_date
0,CUST_00038,Maria Smith,maria.smith@email.com,419-689-7149,San Antonio,2023-07-21
1,CUST_00068,Merie Devis,mariadavis@emailcom,398-133-4792,Phoenix,2024-03-18
2,CUST_00007,Micheel Mertinez,michaelmartinez@emailcom,682-408-7653,Dallas,2023-11-09
3,CUST_00049,Sarah Miller,sarah.miller@email.com,600-668-4937,San Diego,2024-05-19
4,CUST_00080,Robert Davis,robert.davis@email.com,395-778-7561,Dallas,2026-02-02


In [7]:
print("🔍 LEGACY SYSTEM - First 5 rows:")
df_legacy.head()

🔍 LEGACY SYSTEM - First 5 rows:


,customer_id,full_name,email,phone,city,signup_date
0,CUST_00000,David Williams,david.williams@email.com,694-371-1626,Los Angeles,2024-07-09
1,CUST_00001,Robert Garcia,robert.garcia@email.com,846-905-6139,San Antonio,2024-06-29
2,CUST_00002,William Johnson,william.johnson@email.com,494-690-4114,Phoenix,2026-01-25
3,CUST_00003,Robert Smith,robert.smith@email.com,632-925-9821,Houston,2025-03-04
4,CUST_00004,Robert Johnson,robert.johnson@email.com,780-438-6143,Los Angeles,2024-03-04


##  Diagnosing the Problem

Let's analyze what kinds of duplicates we have:
1. **Exact duplicates** - Same name, email, phone (easy!)
2. **Near duplicates** - Slight variations in name format
3. **Typo duplicates** - "John Smith" vs "Jon Smith"
4. **Format differences** - "John Smith" vs "Smith, John"

In [8]:
# First, let's check for exact duplicates WITHIN each system

# Check main CRM for duplicates
main_dupes = df_main[df_main.duplicated(subset=['full_name', 'email'], keep=False)]
print(f"Exact duplicates in MAIN CRM: {len(main_dupes)} rows")

# Check legacy system for duplicates
legacy_dupes = df_legacy[df_legacy.duplicated(subset=['full_name', 'email'], keep=False)]
print(f"Exact duplicates in LEGACY system: {len(legacy_dupes)} rows")

Exact duplicates in MAIN CRM: 89 rows
Exact duplicates in LEGACY system: 66 rows


##  Step 2: Find Exact Matches First

Always start with exact matches - they're 100% reliable.